# NB06 — Part Validation

Validates an exported STL against engineering acceptance criteria.

| Input | Source |
|---|---|
| `outputs/stl/{part}_optimized.stl` | NB05 export |
| `outputs/meshes/{part}_stage04.json` | SIMP sidecar (compliance, VF, grid) |
| `scad/{part}_params.json` | geometry, load case, material |

**Two validation modes**
- `fast` — compliance-based safety factor + overhang + feature size + dimensional check (< 2 min)
- `full` — all of the above + gmsh STL→tet remesh + FEniCSx FEA with original BCs (10–20 min)

> ⚠  **Euler number is NOT a quality gate here.** Internal structural handles produced by the
> topology optimizer are correct load-bearing structure. A watertight STL with 0 open edges
> and a converged SIMP run is a valid part regardless of its Euler number.


In [ ]:
# ── Papermill parameters ──────────────────────────────────────────────────────
PART_NAME_OVERRIDE = ""
VALIDATION_MODE = "full"
BUILD_DIRECTION = "Z"
OVERHANG_ANGLE_DEG = 45.0
FEA_MESH_SIZE_FACTOR = 2.0
SF_THRESHOLD = 2.0
STRESS_CONC_FACTOR = 3.0

In [ ]:
import json
import sys
import time
import datetime
from pathlib import Path

import numpy as np
import trimesh
import h5py
from scipy.ndimage import distance_transform_edt
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# ── Workspace layout ──────────────────────────────────────────────────────────
WORKSPACE   = Path("/workspace")
SCAD_DIR    = WORKSPACE / "scad"
OUTPUT_ROOT = WORKSPACE / "outputs"
STL_DIR     = OUTPUT_ROOT / "stl"
MESH_DIR    = OUTPUT_ROOT / "meshes"
REPORT_DIR  = OUTPUT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ── Resolve part name ─────────────────────────────────────────────────────────
if PART_NAME_OVERRIDE:
    part_name = PART_NAME_OVERRIDE
else:
    state_path = OUTPUT_ROOT / "pipeline_state.json"
    if state_path.exists():
        with open(state_path) as _f:
            part_name = json.load(_f)["PART_NAME"]
    else:
        raise RuntimeError(
            "No PART_NAME_OVERRIDE and no outputs/pipeline_state.json. "
            "Pass -p PART_NAME_OVERRIDE <name> to Papermill."
        )

print(f"Part: {part_name}  |  Mode: {VALIDATION_MODE}  |  Build dir: {BUILD_DIRECTION}")

# ── Key paths ─────────────────────────────────────────────────────────────────
params_path  = SCAD_DIR    / f"{part_name}_params.json"
stage04_path = MESH_DIR    / f"{part_name}_stage04.json"
stage04_h5   = MESH_DIR    / f"{part_name}_stage04.h5"
stl_path     = STL_DIR     / f"{part_name}_optimized.stl"
msh_path     = MESH_DIR    / f"{part_name}_validation.msh"
report_path  = REPORT_DIR  / f"{part_name}_validation.json"

for _p in [params_path, stage04_path, stl_path]:
    assert _p.exists(), f"Required input missing: {_p}"
    print(f"  ✓ {_p.relative_to(WORKSPACE)}")

# ── Pre-initialise variables consumed by the sign-off cell ───────────────────
sf_compliance      = None
sigma_rms_MPa      = None
sf_compliance_status = "SKIPPED"

sf_fea             = None
sigma_max_fea      = None
sf_fea_status      = "SKIPPED"

overhang_pct       = None
overhang_area_mm2  = None
max_overhang_deg   = None

min_thickness_mm   = None
pct5_thickness_mm  = None
feature_pct_below  = None

dim_errors         = {}
actual_dims        = np.zeros(3)


In [ ]:
# ── params.json ───────────────────────────────────────────────────────────────
with open(params_path) as _f:
    params = json.load(_f)

# ── stage04 sidecar ───────────────────────────────────────────────────────────
with open(stage04_path) as _f:
    stage04 = json.load(_f)

for _field in ["final_compliance", "volume_fraction", "converged"]:
    if _field not in stage04:
        print(f"  ⚠  stage04 missing '{_field}' — will use None")

final_compliance = stage04.get("final_compliance")  # N·mm
volume_fraction  = stage04.get("volume_fraction") or stage04.get("final_volume_frac")
converged        = stage04.get("converged", False)
voxel_size_mm    = float(
    stage04.get("voxel_size_mm")
    or stage04.get("grid", {}).get("voxel_size", 0) * 1000.0  # SI→mm
    or params.get("simp", {}).get("voxel_size_mm")
    or params.get("simp", {}).get("VOXEL_SIZE_MM")
    or 1.0
)

# ── Material ──────────────────────────────────────────────────────────────────
mat = params.get("material", {})
E_Pa            = float(mat.get("E", 210e9))
E_MPa           = E_Pa / 1e6
nu              = float(mat.get("nu", 0.3))
sigma_yield_raw = mat.get("sigma_yield", mat.get("yield_strength", 250e6))
sigma_yield_MPa = float(sigma_yield_raw) / (1e6 if float(sigma_yield_raw) > 1e4 else 1.0)

# ── Load case ─────────────────────────────────────────────────────────────────
_lc = (params.get("load_cases") or [{}])[0]
force_vector = np.array(
    _lc.get("force", _lc.get("force_vector", [0.0, 0.0, -1000.0])),
    dtype=float,
)
F_total = float(np.linalg.norm(force_vector))

print(f"Material : E={E_MPa:.0f} MPa  ν={nu}  σ_yield={sigma_yield_MPa:.0f} MPa")
print(f"Load     : F={F_total:.1f} N  vector={force_vector.tolist()}")
print(f"SIMP     : C={final_compliance}  VF={volume_fraction}  converged={converged}  voxel={voxel_size_mm}mm")

# ── STL ───────────────────────────────────────────────────────────────────────
mesh_tri = trimesh.load(str(stl_path), force="mesh")
assert isinstance(mesh_tri, trimesh.Trimesh), "trimesh.load did not return a Trimesh"

stl_watertight = mesh_tri.is_watertight
_es = np.sort(mesh_tri.edges, axis=1)
_eu, _ec = np.unique(_es, axis=0, return_counts=True)
stl_open_edges = int((_ec == 1).sum())
stl_volume_mm3 = float(mesh_tri.volume) if stl_watertight else None
euler          = mesh_tri.euler_number
n_faces        = len(mesh_tri.faces)
n_verts        = len(mesh_tri.vertices)

print(f"\nSTL Quality")
print(f"  Watertight   : {stl_watertight}")
print(f"  Open edges   : {stl_open_edges}")
print(f"  Euler #      : {euler}  (internal structural handles are expected — not a defect)")
print(f"  Vertices     : {n_verts:,}")
print(f"  Faces        : {n_faces:,}")
if stl_volume_mm3 is not None:
    print(f"  Volume       : {stl_volume_mm3:.0f} mm³")
else:
    print("  Volume       : N/A (not watertight)")

# ── Geometry bounding box for BC detection ────────────────────────────────────
geom  = params.get("geometry", {})
bbox  = mesh_tri.bounds  # [[x,y,z]_min, [x,y,z]_max]
bc_tol = 2.0 * voxel_size_mm

def _g(*keys, default=None):
    for k in keys:
        if k in geom:
            return float(geom[k])
    return default

target_dims = np.array([
    _g("LENGTH", "length", "L"),
    _g("WIDTH",  "width",  "W"),
    _g("HEIGHT", "height", "H"),
], dtype=object)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Compliance-Based Safety Factor — fast gate, always runs
#
# Method: strain-energy density approach
#   Strain energy  U = C/2   (compliance C = 2U = u^T K u)
#   Energy density U/V = σ_rms² / (2E)
#   → σ_rms = sqrt(E · C / V)
#
# A conservative stress-concentration factor Kt is applied:
#   σ_est = Kt · σ_rms
#   SF    = σ_yield / σ_est
#
# This is a *lower-bound* estimate — σ_rms < σ_max.
# FEA re-mesh (Cell 6) computes σ_max rigorously.
# ─────────────────────────────────────────────────────────────────────────────

if final_compliance is not None and stl_volume_mm3 is not None:
    # Solver outputs compliance in SI (N·m). Convert to N·mm for consistency
    # with E_MPa [N/mm²] and V_mm3 [mm³].
    C_Nmm = float(final_compliance) * 1000.0
    V_mm3         = stl_volume_mm3
    sigma_rms_MPa = float(np.sqrt(E_MPa * C_Nmm / V_mm3))
    sigma_est_MPa = sigma_rms_MPa * STRESS_CONC_FACTOR
    sf_compliance = sigma_yield_MPa / sigma_est_MPa
    sf_compliance_status = "PASS" if sf_compliance >= SF_THRESHOLD else "FAIL"

    print("Compliance-Based Safety Factor (fast gate)")
    print(f"  C           = {C_Nmm:.3f} N·mm")
    print(f"  V_solid     = {V_mm3:.0f} mm³")
    print(f"  E           = {E_MPa:.0f} MPa")
    print(f"  σ_rms       = {sigma_rms_MPa:.2f} MPa  [energy density estimate]")
    print(f"  σ_est       = {sigma_est_MPa:.2f} MPa  [Kt = {STRESS_CONC_FACTOR:.1f}]")
    print(f"  SF          = {sf_compliance:.2f}  [{sf_compliance_status}]  (threshold ≥ {SF_THRESHOLD})")
    print()
    print("  ⚠  σ_rms underestimates σ_max by definition.")
    print("     Kt={:.1f} is a conservative stand-in; FEA re-mesh gives σ_max.".format(STRESS_CONC_FACTOR))
    if sf_compliance_status == "FAIL":
        print(f"\n  ✗ FAST-GATE FAILED — SF={sf_compliance:.2f} < threshold {SF_THRESHOLD}")
        print("    This is a conservative estimate. Proceed to FEA re-mesh before rejecting.")
else:
    missing = []
    if final_compliance is None:
        missing.append("final_compliance (stage04.json)")
    if stl_volume_mm3 is None:
        missing.append("STL volume (STL not watertight)")
    print(f"⚠  Compliance-based SF skipped — missing: {', '.join(missing)}")


In [ ]:
if VALIDATION_MODE != "full":
    print(f"VALIDATION_MODE='{VALIDATION_MODE}': skipping re-mesh and FEA")
else:
    try:
        import tetgen
        import basix.ufl
        import dolfinx.mesh as dmesh_mod
        from dolfinx.io import gmshio

        print(f"TetGen {tetgen.__version__} — tet mesh from watertight STL")
        t0 = time.perf_counter()

        # Repair surface mesh before TetGen — marching cubes output often has
        # near-coincident vertices and degenerate faces that cause recoversubfaces.
        mesh_repair = mesh_tri.copy()
        trimesh.repair.fix_winding(mesh_repair)
        trimesh.repair.fix_normals(mesh_repair)
        mesh_repair.merge_vertices(merge_tex=False)
        mesh_repair.remove_degenerate_faces()
        mesh_repair.remove_duplicate_faces()
        print(f"  Repaired STL: {len(mesh_repair.vertices):,} verts  "
              f"{len(mesh_repair.faces):,} faces  "
              f"watertight={mesh_repair.is_watertight}")

        tet = tetgen.TetGen(mesh_repair.vertices, mesh_repair.faces)
        nodes, elems = tet.tetrahedralize(
            order   = 1,
            quality = False,
            verbose = False,
        )
        n_tets  = elems.shape[0]
        n_nodes = nodes.shape[0]
        print(f"  Nodes       : {n_nodes:,}")
        print(f"  Tetrahedra  : {n_tets:,}")
        print(f"  Mesh time   : {time.perf_counter() - t0:.1f}s")

        _element = basix.ufl.element("Lagrange", "tetrahedron", 1, shape=(3,))
        _domain  = ufl.Mesh(_element)
        msh_obj  = dmesh_mod.create_mesh(
            MPI.COMM_WORLD,
            elems.astype(np.int64),
            nodes.astype(np.float64),
            _domain,
        )
        msh_obj.topology.create_connectivity(2, 3)
        _gmsh_ok = True
        print(f"  dolfinx mesh built in memory")

    except Exception as _e:
        import traceback
        print(f"  ✗ TetGen failed: {_e}")
        traceback.print_exc()
        print("    Degrading to VALIDATION_MODE='fast'")
        VALIDATION_MODE = "fast"

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEniCSx FEA on re-meshed STL geometry
#
# BC detection strategy (in order of precedence):
#   1. params.json load_cases[0] explicit keys: "fixed_face", "load_face"
#   2. Spatial inference: fixed face = face closest to origin along force axis;
#      load face = opposite face.
#
# Traction is applied as F_total / A_load on the load facets.
# ─────────────────────────────────────────────────────────────────────────────

if VALIDATION_MODE != "full" or not _gmsh_ok:
    print("FEA re-mesh skipped.")
else:
    import dolfinx
    from dolfinx import fem, mesh as dmesh, default_scalar_type
    from dolfinx.io import gmshio
    import dolfinx.fem.petsc as petsc_fem
    import ufl
    from mpi4py import MPI
    from petsc4py import PETSc

    print(f"FEniCSx {dolfinx.__version__} — FEA on {msh_path.name}")
    t0 = time.perf_counter()

    # ── Load mesh ─────────────────────────────────────────────────────────────
    # msh_obj built in memory by Cell 5 (TetGen) — no file read needed
    fea_n_cells = msh_obj.topology.index_map(3).size_local
    fea_n_nodes = msh_obj.topology.index_map(0).size_local
    print(f"  Cells: {fea_n_cells:,}  Nodes: {fea_n_nodes:,}")

    # ── Function space (P1 displacement) ──────────────────────────────────────
    V = fem.functionspace(msh_obj, ("Lagrange", 1, (3,)))

    # ── BC region detection ───────────────────────────────────────────────────
    _lc     = (params.get("load_cases") or [{}])[0]
    _bb_min = msh_obj.geometry.x.min(axis=0)
    _bb_max = msh_obj.geometry.x.max(axis=0)

    # Parse explicit face keys, or infer from force direction
    _fixed_face = _lc.get("fixed_face", "")
    _load_face  = _lc.get("load_face", "")
    _tol        = bc_tol  # 2 × voxel_size_mm

    _FACE_MAP = {
        "x_min": (0, _bb_min[0]), "x_max": (0, _bb_max[0]),
        "y_min": (1, _bb_min[1]), "y_max": (1, _bb_max[1]),
        "z_min": (2, _bb_min[2]), "z_max": (2, _bb_max[2]),
        "bottom": (2, _bb_min[2]), "top": (2, _bb_max[2]),
        "left":  (0, _bb_min[0]), "right": (0, _bb_max[0]),
        "front": (1, _bb_min[1]), "back":  (1, _bb_max[1]),
    }

    def _make_plane_fn(axis_idx, val, tol):
        def _fn(x):
            return np.isclose(x[axis_idx], val, atol=tol)
        return _fn

    if _fixed_face.lower() in _FACE_MAP:
        _ax, _val = _FACE_MAP[_fixed_face.lower()]
        fixed_fn = _make_plane_fn(_ax, _val, _tol)
        print(f"  Fixed BC  : {_fixed_face} (axis {_ax} = {_val:.2f} ± {_tol:.2f})")
    else:
        # Infer: fixed face is on the side opposite to force application
        _fi = int(np.argmax(np.abs(force_vector)))  # dominant force axis
        _fv = _bb_min[_fi] if force_vector[_fi] < 0 else _bb_max[_fi]
        fixed_fn = _make_plane_fn(_fi, _fv, _tol)
        print(f"  Fixed BC  : inferred axis={_fi}, value={_fv:.2f}mm (force axis heuristic)")

    if _load_face.lower() in _FACE_MAP:
        _ax, _val = _FACE_MAP[_load_face.lower()]
        load_fn = _make_plane_fn(_ax, _val, _tol)
        print(f"  Load BC   : {_load_face} (axis {_ax} = {_val:.2f} ± {_tol:.2f})")
    else:
        _li = int(np.argmax(np.abs(force_vector)))
        _lv = _bb_max[_li] if force_vector[_li] < 0 else _bb_min[_li]
        load_fn = _make_plane_fn(_li, _lv, _tol)
        print(f"  Load BC   : inferred axis={_li}, value={_lv:.2f}mm (opposite fixed face)")

    # ── Dirichlet BC (zero displacement on fixed face) ────────────────────────
    fixed_dofs = fem.locate_dofs_geometrical(V, fixed_fn)
    u_zero = np.zeros(3, dtype=default_scalar_type)
    bc = fem.dirichletbc(u_zero, fixed_dofs, V)
    print(f"  Fixed DOFs: {len(fixed_dofs)}")

    # ── Neumann BC — traction on load face ────────────────────────────────────
    load_facets = dmesh.locate_entities_boundary(msh_obj, 2, load_fn)
    print(f"  Load facets: {len(load_facets)}")
    if len(load_facets) == 0:
        raise RuntimeError(
            "No load facets detected. Check 'load_face' in params.json or "
            "increase bc_tol (currently {:.2f}mm).".format(_tol)
        )

    mt_load = dmesh.meshtags(
        msh_obj, 2,
        load_facets.astype(np.int32),
        np.ones(len(load_facets), dtype=np.int32),
    )
    ds_load = ufl.Measure("ds", domain=msh_obj, subdomain_data=mt_load)

    # Compute load face area, then traction
    _one = fem.Constant(msh_obj, default_scalar_type(1.0))
    load_area_mm2 = float(fem.assemble_scalar(fem.form(_one * ds_load(1))))
    traction_MPa  = force_vector / load_area_mm2
    print(f"  Load area   : {load_area_mm2:.2f} mm²")
    print(f"  Traction    : {traction_MPa} N/mm² (MPa)")

    # ── Linear elasticity variational form (all quantities in MPa/mm) ─────────
    mu_v  = fem.Constant(msh_obj, default_scalar_type(E_MPa / (2.0 * (1.0 + nu))))
    lam_v = fem.Constant(msh_obj, default_scalar_type(E_MPa * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))))

    def eps(u):
        return ufl.sym(ufl.grad(u))

    def sigma_el(u):
        return lam_v * ufl.tr(eps(u)) * ufl.Identity(3) + 2 * mu_v * eps(u)

    u_tr, v_tr = ufl.TrialFunction(V), ufl.TestFunction(V)
    a = ufl.inner(sigma_el(u_tr), eps(v_tr)) * ufl.dx

    t_vec = fem.Constant(msh_obj, traction_MPa.astype(default_scalar_type))
    L = ufl.inner(t_vec, v_tr) * ds_load(1)

    # ── Solve (CG + GAMG — scales well for > 100k DOFs) ──────────────────────
    problem = petsc_fem.LinearProblem(
        a, L, bcs=[bc],
        petsc_options={
            "ksp_type":        "cg",
            "pc_type":         "gamg",
            "ksp_rtol":        1e-8,
            "ksp_max_it":      2000,
            "ksp_monitor":     "",          # suppress iteration output
        },
    )
    u_h = problem.solve()
    print(f"  Solve time  : {time.perf_counter() - t0:.1f}s")

    # ── Von Mises stress field (DG0 — element-wise) ───────────────────────────
    S = fem.functionspace(msh_obj, ("DG", 0))
    dev_s  = ufl.dev(sigma_el(u_h))
    vm_ufl = ufl.sqrt((3.0 / 2.0) * ufl.inner(dev_s, dev_s))
    vm_fn  = fem.Function(S, name="VonMises_MPa")
    vm_fn.interpolate(fem.Expression(vm_ufl, S.element.interpolation_points()))

    # Gather across MPI ranks (single-rank here, but correct either way)
    vm_all = vm_fn.x.array
    sigma_max_fea  = float(msh_obj.comm.allreduce(vm_all.max(),  op=MPI.MAX))
    sigma_mean_fea = float(msh_obj.comm.allreduce(vm_all.mean(), op=MPI.SUM) / msh_obj.comm.size)

    sf_fea = sigma_yield_MPa / sigma_max_fea
    sf_fea_status = "PASS" if sf_fea >= SF_THRESHOLD else "FAIL"

    # ── Max displacement ──────────────────────────────────────────────────────
    u_mag = np.linalg.norm(u_h.x.array.reshape(-1, 3), axis=1)
    u_max = float(msh_obj.comm.allreduce(u_mag.max(), op=MPI.MAX))

    print(f"\nFEA Results")
    print(f"  σ_max (VM)  : {sigma_max_fea:.2f} MPa")
    print(f"  σ_mean (VM) : {sigma_mean_fea:.2f} MPa")
    print(f"  u_max       : {u_max:.4f} mm")
    print(f"  SF          : {sf_fea:.2f}  [{sf_fea_status}]  (threshold ≥ {SF_THRESHOLD})")
    print(f"  Total time  : {time.perf_counter() - t0:.1f}s")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Overhang Analysis
#
# A face needs AM support if:
#   dot(face_normal, build_dir) < -cos(OVERHANG_ANGLE_DEG)
#
# OVERHANG_ANGLE_DEG is measured from horizontal (typical: 45°).
# For 45°: threshold_dot = -cos(45°) ≈ -0.707
#   dot = -1.0 → flat underside (always needs support)
#   dot = -0.707 → exactly at threshold (self-supporting limit)
#   dot =  0.0 → vertical wall (self-supporting)
# ─────────────────────────────────────────────────────────────────────────────

_bd_map = {
    "X": np.array([1., 0., 0.]),
    "Y": np.array([0., 1., 0.]),
    "Z": np.array([0., 0., 1.]),
}
build_vec = _bd_map[BUILD_DIRECTION.upper()]

normals    = mesh_tri.face_normals   # (n_faces, 3) outward unit normals
areas      = mesh_tri.area_faces     # (n_faces,) mm²
dot_bd     = normals @ build_vec     # (n_faces,) in [-1, 1]

threshold_dot = -np.cos(np.deg2rad(OVERHANG_ANGLE_DEG))
overhang_mask = dot_bd < threshold_dot

# For overhanging faces: angle from horizontal = arcsin(|dot|) when dot < 0
face_angles = np.where(
    dot_bd < 0,
    np.degrees(np.arcsin(np.clip(-dot_bd, 0.0, 1.0))),
    0.0,
)

n_overhang_faces  = int(overhang_mask.sum())
overhang_area_mm2 = float(areas[overhang_mask].sum())
total_area_mm2    = float(areas.sum())
overhang_pct      = 100.0 * overhang_area_mm2 / total_area_mm2
max_overhang_deg  = float(face_angles[overhang_mask].max()) if n_overhang_faces > 0 else 0.0

# Histogram of face angles for overhanging faces
print(f"Overhang Analysis  (build: {BUILD_DIRECTION}, threshold: {OVERHANG_ANGLE_DEG}°)")
print(f"  Overhanging faces  : {n_overhang_faces:,} / {len(normals):,}")
print(f"  Overhanging area   : {overhang_area_mm2:.1f} mm²  ({overhang_pct:.1f}% of total surface)")
if n_overhang_faces > 0:
    print(f"  Max overhang angle : {max_overhang_deg:.1f}°  (from horizontal)")
    # Angle distribution for overhanging faces
    _angle_bins = [45, 60, 75, 90]
    _prev = OVERHANG_ANGLE_DEG
    for _b in _angle_bins:
        _cnt = ((face_angles >= _prev) & (face_angles < _b) & overhang_mask).sum()
        _a   = areas[(face_angles >= _prev) & (face_angles < _b) & overhang_mask].sum()
        print(f"    [{_prev:.0f}°–{_b:.0f}°)  {_cnt:5d} faces  {_a:8.1f} mm²")
        _prev = _b
else:
    print("  ✓ No overhanging faces detected")

# ── Plot: histogram of angle distribution across all faces ────────────────────
_all_angles = np.degrees(np.arccos(np.clip(np.abs(dot_bd), 0, 1)))  # 0=horiz, 90=vert
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(_all_angles, bins=45, range=(0, 90), color="#4c72b0", alpha=0.8, edgecolor="none", label="All faces")
ax.hist(_all_angles[overhang_mask], bins=45, range=(0, 90), color="#c44e52", alpha=0.7, edgecolor="none", label="Needs support")
ax.axvline(OVERHANG_ANGLE_DEG, color="black", linestyle="--", linewidth=1.2, label=f"Threshold ({OVERHANG_ANGLE_DEG}°)")
ax.set_xlabel("Face angle from horizontal (°)")
ax.set_ylabel("Face count")
ax.set_title(f"{part_name} — Overhang distribution (build: {BUILD_DIRECTION})")
ax.legend(fontsize=8)
plt.tight_layout()
_fig_path = REPORT_DIR / f"{part_name}_overhang.png"
plt.savefig(str(_fig_path), dpi=130)
plt.show()
print(f"  Plot saved: {_fig_path.name}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Feature Size Distribution
#
# Method: 3D distance transform on the SIMP density field.
#   binary[i,j,k] = density[i,j,k] > DENSITY_THRESHOLD
#   dt[i,j,k]     = distance to nearest void (mm)
#   local_thickness = 2 × dt  (inscribed sphere diameter)
#
# This captures the true internal topology (including structural ribs)
# without any surface-mesh approximation.  Requires stage04.h5.
# ─────────────────────────────────────────────────────────────────────────────
density_3d = None

density_npy = MESH_DIR / f"{part_name}_density.npy"
if density_npy.exists():
    density_3d = np.load(str(density_npy))
    print(f"Density field: {density_npy.name}  shape={density_3d.shape}  voxel={voxel_size_mm}mm")
else:
    print(f"⚠  {density_npy.name} not found — skipping feature-size analysis.")
    print("   (NB04 writes this as outputs/meshes/{part_name}_density.npy)")

if density_3d is not None:
    _dt  = float(params.get("simp", {}).get("DENSITY_THRESHOLD", 0.45))
    binary = density_3d > _dt

    n_solid  = int(binary.sum())
    n_total  = int(binary.size)
    actual_vf = n_solid / n_total
    print(f"  Solid voxels: {n_solid:,} / {n_total:,}  (VF={actual_vf:.3f})")

    # Distance transform — sampling accounts for anisotropic voxels if needed
    dt = distance_transform_edt(binary, sampling=(voxel_size_mm,) * 3)
    local_thickness = 2.0 * dt[binary]  # mm

    min_thickness_mm  = float(local_thickness.min())
    pct5_thickness_mm = float(np.percentile(local_thickness, 5))
    pct50             = float(np.percentile(local_thickness, 50))
    pct95             = float(np.percentile(local_thickness, 95))

    # Common AM process limits (for reference; not hard gates here)
    _FDM_MIN  = 0.8   # mm (conservative — 2× typical 0.4mm nozzle)
    _SLA_MIN  = 0.3   # mm
    _SLS_MIN  = 0.6   # mm

    feature_pct_below = float((local_thickness < _FDM_MIN).mean() * 100)

    print(f"\nFeature Size Distribution (local thickness from distance transform)")
    print(f"  Min              : {min_thickness_mm:.2f} mm")
    print(f"  5th percentile   : {pct5_thickness_mm:.2f} mm")
    print(f"  Median           : {pct50:.2f} mm")
    print(f"  95th percentile  : {pct95:.2f} mm")
    print(f"  < {_FDM_MIN}mm (FDM min)  : {feature_pct_below:.1f}% of solid voxels")
    print(f"  < {_SLS_MIN}mm (SLS min)  : {(local_thickness < _SLS_MIN).mean()*100:.1f}% of solid voxels")
    print(f"  < {_SLA_MIN}mm (SLA min)  : {(local_thickness < _SLA_MIN).mean()*100:.1f}% of solid voxels")

    # ── Histogram ─────────────────────────────────────────────────────────────
    _clip_max = float(np.percentile(local_thickness, 99))  # clip outliers for vis
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(np.clip(local_thickness, 0, _clip_max), bins=60,
            color="#2ca02c", alpha=0.8, edgecolor="none")
    for _proc, _lim, _col in [("FDM", _FDM_MIN, "red"),
                                ("SLS", _SLS_MIN, "orange"),
                                ("SLA", _SLA_MIN, "purple")]:
        ax.axvline(_lim, color=_col, linestyle="--", linewidth=1.0, label=f"{_proc} min ({_lim}mm)")
    ax.set_xlabel("Local Thickness (mm)")
    ax.set_ylabel("Voxel Count")
    ax.set_title(f"{part_name} — Feature Size Distribution")
    ax.legend(fontsize=8)
    plt.tight_layout()
    _fig_path = REPORT_DIR / f"{part_name}_feature_size.png"
    plt.savefig(str(_fig_path), dpi=130)
    plt.show()
    print(f"  Plot saved: {_fig_path.name}")


NameError: name 'MESH_DIR' is not defined

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Dimensional Check — STL bounding box vs target envelope from params.json
#
# The topology optimizer fills a design domain; the STL should fit within the
# original geometry envelope.  A 5% tolerance is used as a soft warning.
#
# Note: topology-optimized parts will be SMALLER than the envelope (material
# removed) so under-size in each axis is expected and not a failure.
# Over-size indicates a mesh export error.
# ─────────────────────────────────────────────────────────────────────────────
bounds     = mesh_tri.bounds
actual_dims = bounds[1] - bounds[0]

print(f"Dimensional Check — {part_name}")
print(f"  STL bounds  : [{bounds[0][0]:.2f}, {bounds[0][1]:.2f}, {bounds[0][2]:.2f}] "
      f"→ [{bounds[1][0]:.2f}, {bounds[1][1]:.2f}, {bounds[1][2]:.2f}] mm")
print(f"  Bounding box: {actual_dims[0]:.2f} × {actual_dims[1]:.2f} × {actual_dims[2]:.2f} mm")

_dim_labels = ["Length (X)", "Width  (Y)", "Height (Z)"]
dim_errors  = {}

for _i, (_label, _target) in enumerate(zip(_dim_labels, target_dims)):
    _actual = float(actual_dims[_i])
    if _target is not None:
        _target = float(_target)
        _err_pct = (_actual - _target) / _target * 100.0
        _over  = _err_pct > 0
        if abs(_err_pct) < 5.0:
            _status = "✓ OK"
        elif _over:
            _status = "✗ OVER-SIZE"  # unexpected — flag as potential export error
        else:
            _status = "~ UNDER-SIZE"  # expected for TO output
        print(f"  {_label}: target={_target:.2f}  actual={_actual:.2f}  "
              f"Δ={_err_pct:+.1f}%  {_status}")
        dim_errors[_label.strip()] = {
            "target_mm":  _target,
            "actual_mm":  _actual,
            "error_pct":  round(float(_err_pct), 2),
            "status":     "FAIL" if _over and abs(_err_pct) >= 5.0 else "PASS",
        }
    else:
        print(f"  {_label}: target=N/A (not in params.json)  actual={_actual:.2f}mm")

# Centroid check — should be near expected centre of design domain
centroid = mesh_tri.centroid
if target_dims[0] is not None and target_dims[1] is not None and target_dims[2] is not None:
    _expected_centre = np.array([float(t) / 2.0 for t in target_dims])
    _centroid_offset = centroid - _expected_centre
    print(f"\n  Centroid    : {centroid}  (expected ~{_expected_centre})")
    print(f"  Offset      : {_centroid_offset}  (norm={np.linalg.norm(_centroid_offset):.2f}mm)")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Sign-Off Report
# Assembles all checks into a structured JSON sidecar.
# ─────────────────────────────────────────────────────────────────────────────
_dim_overall = (
    "FAIL" if any(v.get("status") == "FAIL" for v in dim_errors.values())
    else "WARN" if any(v.get("error_pct", 0) > 2.0 for v in dim_errors.values())
    else "PASS" if dim_errors
    else "SKIPPED"
)

validation_report = {
    "part_name"       : part_name,
    "timestamp"       : datetime.datetime.utcnow().isoformat() + "Z",
    "validation_mode" : VALIDATION_MODE,

    "pipeline_state": {
        "converged"        : converged,
        "volume_fraction"  : volume_fraction,
        "final_compliance" : final_compliance,
        "voxel_size_mm"    : voxel_size_mm,
    },

    "stl_quality": {
        "watertight"    : stl_watertight,
        "open_edges"    : stl_open_edges,
        "euler_number"  : euler,
        "volume_mm3"    : stl_volume_mm3,
        "face_count"    : n_faces,
        "vertex_count"  : n_verts,
        "note"          : "Euler ≠ 0 is expected for topology-optimized parts with internal structural handles.",
        "status"        : "PASS" if stl_watertight and stl_open_edges == 0 else "FAIL",
    },

    "stress_analysis": {
        "material": {
            "E_MPa"         : E_MPa,
            "nu"            : nu,
            "sigma_yield_MPa": sigma_yield_MPa,
        },
        "applied_load_N"   : float(F_total),
        "force_vector_N"   : force_vector.tolist(),

        "compliance_based": {
            "method"         : "σ_rms = sqrt(E·C/V), SF = σ_yield / (Kt·σ_rms)",
            "C_Nmm"          : float(final_compliance) if final_compliance else None,
            "sigma_rms_MPa"  : round(sigma_rms_MPa, 2) if sigma_rms_MPa else None,
            "Kt"             : STRESS_CONC_FACTOR,
            "sigma_est_MPa"  : round(sigma_rms_MPa * STRESS_CONC_FACTOR, 2) if sigma_rms_MPa else None,
            "SF"             : round(sf_compliance, 2) if sf_compliance else None,
            "SF_threshold"   : SF_THRESHOLD,
            "status"         : sf_compliance_status,
            "note"           : "Conservative lower-bound. σ_rms < σ_max by definition.",
        },

        "fea_remesh": {
            "method"         : "gmsh STL→tet + FEniCSx FEA + von Mises",
            "sigma_max_MPa"  : round(sigma_max_fea, 2) if sigma_max_fea else None,
            "SF"             : round(sf_fea, 2) if sf_fea else None,
            "SF_threshold"   : SF_THRESHOLD,
            "status"         : sf_fea_status,
        },
    },

    "printability": {
        "build_direction"      : BUILD_DIRECTION,
        "overhang_threshold_deg": OVERHANG_ANGLE_DEG,
        "overhanging_area_mm2" : round(overhang_area_mm2, 1) if overhang_area_mm2 is not None else None,
        "overhanging_area_pct" : round(overhang_pct, 2) if overhang_pct is not None else None,
        "max_overhang_angle_deg": round(max_overhang_deg, 1) if max_overhang_deg is not None else None,
        "min_feature_mm"       : round(min_thickness_mm, 3) if min_thickness_mm is not None else None,
        "feature_pct5_mm"      : round(pct5_thickness_mm, 3) if pct5_thickness_mm is not None else None,
        "pct_below_fdm_min"    : round(feature_pct_below, 2) if feature_pct_below is not None else None,
        "status"               : "INFO",  # informational — no hard threshold
    },

    "dimensional_accuracy": {
        "actual_dims_mm"   : [round(float(d), 3) for d in actual_dims],
        "target_dims_mm"   : [round(float(t), 3) if t is not None else None for t in target_dims],
        "dimension_errors" : dim_errors,
        "status"           : _dim_overall,
    },

    "overall_status": None,
}

# ── Compute overall ───────────────────────────────────────────────────────────
_checks = [
    validation_report["stl_quality"]["status"],
    validation_report["stress_analysis"]["compliance_based"]["status"],
    validation_report["stress_analysis"]["fea_remesh"]["status"],
    validation_report["dimensional_accuracy"]["status"],
]
if "FAIL" in _checks:
    _overall = "FAIL"
elif "WARN" in _checks:
    _overall = "WARN"
elif all(c in ("PASS", "SKIPPED", "INFO") for c in _checks):
    _overall = "PASS"
else:
    _overall = "INCOMPLETE"

validation_report["overall_status"] = _overall

# ── Write ─────────────────────────────────────────────────────────────────────
with open(report_path, "w") as _f:
    json.dump(validation_report, _f, indent=2)

# ── Summary ───────────────────────────────────────────────────────────────────
_BAR = "=" * 62
print(_BAR)
print(f"  VALIDATION REPORT  —  {part_name}")
print(_BAR)
print(f"  Overall          : {_overall}")
print(f"  STL quality      : {validation_report['stl_quality']['status']}")
_csf = validation_report["stress_analysis"]["compliance_based"]
if _csf["SF"] is not None:
    print(f"  Compliance SF    : {_csf['SF']:.2f}  [{_csf['status']}]  (Kt={STRESS_CONC_FACTOR})")
else:
    print(f"  Compliance SF    : {_csf['status']}")
_fea = validation_report["stress_analysis"]["fea_remesh"]
if _fea["SF"] is not None:
    print(f"  FEA re-mesh SF   : {_fea['SF']:.2f}  [{_fea['status']}]")
else:
    print(f"  FEA re-mesh SF   : {_fea['status']}")
if overhang_pct is not None:
    print(f"  Overhang area    : {overhang_pct:.1f}%  (build: {BUILD_DIRECTION})")
if min_thickness_mm is not None:
    print(f"  Min feature      : {min_thickness_mm:.2f} mm")
print(f"  Dimensional      : {_dim_overall}")
print(_BAR)
print(f"  Report written   : {report_path.relative_to(WORKSPACE)}")
print()

# Render as Markdown table in notebook
_md_rows = [
    ("Overall status",     _overall),
    ("STL watertight",     "✓" if stl_watertight else "✗"),
    ("Open edges",         str(stl_open_edges)),
    ("Euler number",       str(euler)),
    ("Compliance SF",      f"{_csf['SF']:.2f} [{_csf['status']}]" if _csf["SF"] else _csf["status"]),
    ("FEA re-mesh SF",     f"{_fea['SF']:.2f} [{_fea['status']}]" if _fea["SF"] else _fea["status"]),
    ("Overhang area",      f"{overhang_pct:.1f}%" if overhang_pct is not None else "—"),
    ("Min feature (mm)",   f"{min_thickness_mm:.2f}" if min_thickness_mm is not None else "—"),
    ("Dimensional",        _dim_overall),
]
_md = "| Check | Result |\n|---|---|\n"
_md += "\n".join(f"| {k} | {v} |" for k, v in _md_rows)
display(Markdown(_md))
